# 09. Heteroevaluación disponible

**Fuente:** `data/raw/heteroevaluaciondisponible.csv`  
**Salida:** `data/processed/heteroevaluacion_disponible.csv`

Resultados de evaluacion docente por curso/paralelo/periodo (promedio, numero de evaluados). El archivo de origen trae la columna ANIO duplicada por un join previo.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [ ]:
df = pc.leer_csv('heteroevaluaciondisponible.csv', low_memory=False)
df.head()

## 2. Exploración inicial

In [ ]:
pc.resumen(df, 'heteroevaluacion_disponible')

In [ ]:
df.dtypes

## 3. Limpieza

In [ ]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(df)
n_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'Filas duplicadas eliminadas: {n_antes - len(df)}')

## 5. Tipado de fechas e identificadores

In [ ]:
df = pc.castear_enteros(df, ['IDCURSO', 'IDPERSONA'])

## Gráficos exploratorios

Vistas rápidas para apoyar la construcción del catálogo de variables del perfil (Fase 1-2 de la metodología): estacionalidad/tendencia temporal, categorías dominantes y forma de la distribución de las variables numéricas.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)

**Distribución del promedio de heteroevaluación** por curso/paralelo.

In [ ]:
pc.grafico_histograma(df['PROMEDIO'], 'Distribución del promedio de heteroevaluación', bins=30, recorte_percentil=None)

**Heteroevaluaciones registradas por año**.

In [ ]:
conteo_anio = df['ANIO'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(10, 4))
conteo_anio.plot(kind='bar', ax=ax, color=pc.COLOR_PRINCIPAL)
ax.set_title('Heteroevaluaciones registradas por año')
ax.set_xlabel('Año'); ax.set_ylabel('N.º de registros')
plt.tight_layout()

**Unidades con más heteroevaluaciones registradas**.

In [ ]:
pc.grafico_barras(df['UNIDAD'], 'Top 10 unidades por N.º de heteroevaluaciones', top=10)

**Tasa de participación** (evaluados / registrados) por curso: variable derivada útil para medir el compromiso estudiantil detrás de cada promedio.

In [ ]:
tasa_participacion = (df['EVALUADOS'] / df['REGISTRADOS']).replace([float('inf')], pd.NA)
pc.grafico_histograma(tasa_participacion, 'Tasa de participación (evaluados / registrados)', bins=30, recorte_percentil=None)

## 7. Verificación final

In [ ]:
pc.resumen(df, 'heteroevaluacion_disponible (procesado)')
df.head()

## 8. Guardado en data/processed

In [ ]:
pc.guardar_procesado(df, 'heteroevaluacion_disponible.csv')